In [1]:
#LOAD MODEL
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights

weights = ResNet18_Weights.DEFAULT

model = resnet18(weights=weights)

D:\ANACONDA\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Prabhsimrat Singh/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:16<00:00, 2.83MB/s]


In [3]:
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [7]:
import torch.nn as nn
from torchvision import datasets

In [15]:
#FREEZE THE MODEL EXCEPT THE FINAL LAYER
#CHANGE OUTPUT SIZE OF FINAL LAYER
for param in model.parameters():
    param.requires_grad=False

model.fc=nn.Linear(model.fc.in_features,6)

In [57]:
#SEE WHAT PARAMETERS WILL BE CHANGED IN TRAINING OF MODEL
for name,param in model.named_parameters():

    if param.requires_grad:
        print(name)

layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
fc.weight
fc.bias


In [19]:
#OPTIMISER THAT UPDATES PARAMATERS ONLY WHERE REQUIRES_GRAD IS TRUE I\E FOR FINAL LAYER
import torch.optim as optim
optimizer = optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=0.001
)

In [21]:
#INITIALISE WEIGHTS AND TRANFORMERS
weights = ResNet18_Weights.DEFAULT

train_transform = weights.transforms()
test_transform = weights.transforms()

In [33]:
#TRAIN TEST DATABASE
path_train=r"C:\Users\Prabhsimrat Singh\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2\seg_train\seg_train"
path_test=r"C:\Users\Prabhsimrat Singh\.cache\kagglehub\datasets\puneet6060\intel-image-classification\versions\2\seg_test\seg_test"

train_dataset=datasets.ImageFolder(
    root=path_train,
    transform=train_transform
)

test_dataset=datasets.ImageFolder(
    root=path_test,
    transform=train_transform
)


In [37]:
from torch.utils.data import DataLoader
train_loader=DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader=DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [45]:
import torch
device=torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model.to(device)



ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [59]:
criterion=nn.CrossEntropyLoss()

epochs=5
for epoch in range(epochs):

    model.train()

    for images,labels in train_loader:
        images=images.to(device)
        labels=labels.to(device)
        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

In [61]:
model.eval()

y_true=[]
y_pred=[]
with torch.no_grad():

    for x_test,y_test in test_loader:
        x_test=x_test.to(device)
        y_test=y_test.to(device)
        outputs=model(x_test)

        _,predicted=torch.max(outputs,1)

        y_true.extend(y_test.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

In [63]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=test_dataset.classes
    )
)

              precision    recall  f1-score   support

   buildings       0.92      0.93      0.93       437
      forest       0.99      0.99      0.99       474
     glacier       0.91      0.84      0.87       553
    mountain       0.86      0.92      0.89       525
         sea       0.95      0.97      0.96       510
      street       0.95      0.93      0.94       501

    accuracy                           0.93      3000
   macro avg       0.93      0.93      0.93      3000
weighted avg       0.93      0.93      0.93      3000



In [53]:
#STEP 2 FINE TUNING UNFREEZE SOME LAYERS
for param in model.layer4.parameters():
    param.requires_grad = True

In [55]:
optimizer = optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-4
)